# 10 — Gradient Descent Variants

**Difficulty:** ⭐⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 3 hours &nbsp;|&nbsp; **Week 8: Regularisation & Optimisation**

---

## Why This Matters

Every machine learning model you train uses some form of gradient descent under the hood. Linear regression, logistic regression, neural networks, SVMs — they all minimise a loss function by following the gradient downhill. The *variant* you choose directly controls:

- **Speed** — how fast per update and how fast to convergence
- **Noise** — how jittery the parameter path is
- **Memory** — how much data you need to load per step
- **Scalability** — whether it works on a million-row dataset

Understanding the three core variants (Batch, Stochastic, Mini-batch) gives you intuition for every modern optimiser — Adam, RMSProp, AdaGrad — which are all variants of mini-batch GD with adaptive learning rates.

---

## The Analogy: Hiking Down a Mountain

You want to reach the **lowest valley** of a mountain range (the loss minimum). You have three navigation strategies:

### Batch Gradient Descent — Survey the Entire Mountain
Before every single step, you fly a drone over the **entire mountain** and compute the exact downhill direction from your current position. Perfectly accurate direction — but the survey takes hours. For a small mountain (small dataset), this is fine. For a mountain range with a million peaks (large dataset), you'd be paralysed between steps.

### Stochastic Gradient Descent — Look at One Rock
You close your eyes, put your foot down on **one random rock**, feel which way it slopes, and step in that direction. Incredibly fast per step — but noisy. Sometimes the rock slopes slightly uphill by accident. You zigzag but broadly head downward. You never stop exactly at the valley floor; you bounce around it.

### Mini-batch Gradient Descent — Survey a Small Patch
Before each step, you look at a **patch of 32 rocks** around you. Good directional estimate, computed quickly. The path is smoother than one rock, faster than the whole mountain. This is the standard in modern deep learning.

---

## Mathematical Formulation

### Loss function (MSE for regression)
$$\mathcal{L}(\theta) = \frac{1}{n}\sum_{i=1}^{n}(x_i^\top\theta - y_i)^2$$

### Batch GD — full gradient
$$\theta \leftarrow \theta - \eta \cdot \frac{2}{n} X^\top (X\theta - y)$$

### SGD — single-sample gradient
$$\theta \leftarrow \theta - \eta \cdot 2\, x_i(x_i^\top\theta - y_i)$$

### Mini-batch GD — batch gradient
$$\theta \leftarrow \theta - \eta \cdot \frac{2}{B} X_b^\top (X_b\theta - y_b), \quad |b| = B$$

---

## Comparison Table

| Property | Batch GD | SGD | Mini-batch GD |
|---|---|---|---|
| Gradient quality | Exact | Very noisy | Approximate |
| Updates per epoch | 1 | n | n / B |
| Convergence | Smooth, monotone | Noisy, oscillates | Smooth-ish |
| Memory | Full dataset per step | 1 sample per step | B samples per step |
| Best for | Small datasets | Online learning | Deep learning (standard) |
| Parallelisable | Yes (one big batch) | Poor | Yes (GPU-friendly) |

## Section 1 — Setup & House-Price Dataset

We use 500 samples and 5 features for fast convergence demonstration. Features are standardised so all gradient descent variants operate on the same scale.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import time
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ── Reproducibility ──────────────────────────────────────────────────────────
np.random.seed(42)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_palette('tab10')

# ── Synthetic house-price dataset (California-style) ─────────────────────────
N_SAMPLES  = 500
N_FEATURES = 5

# Features: sqft, rooms, age, location_score, income_index
X_raw = np.random.randn(N_SAMPLES, N_FEATURES)

TRUE_COEF  = np.array([2.5, 1.2, -0.7, 1.8, 0.9])
TRUE_BIAS  = 5.0
noise      = np.random.normal(0, 0.5, N_SAMPLES)
y_raw      = X_raw @ TRUE_COEF + TRUE_BIAS + noise

# Standardise X — critical for gradient descent stability
scaler = StandardScaler()
X_std  = scaler.fit_transform(X_raw)

# Add bias column (intercept term) as first column of ones
X = np.hstack([np.ones((N_SAMPLES, 1)), X_std])  # shape (500, 6)
y = y_raw.copy()

feature_names = ['bias', 'sqft', 'rooms', 'age', 'loc_score', 'income']

print(f"Dataset shape (with bias column): {X.shape}")
print(f"Target range: {y.min():.2f} – {y.max():.2f}")
print(f"True coefficients: {TRUE_COEF}  (plus bias={TRUE_BIAS})")

# ── Compute the analytical optimum (OLS solution) for comparison ─────────────
theta_opt = np.linalg.lstsq(X, y, rcond=None)[0]
loss_opt  = np.mean((X @ theta_opt - y) ** 2)
print(f"\nOptimal MSE (OLS):   {loss_opt:.6f}")
print(f"Optimal theta:       {np.round(theta_opt, 3)}")

## Section 2 — Implementing All Three Variants From Scratch

Each function takes `(X, y, lr, n_epochs)`, updates `theta` in-place, records the full-dataset MSE every epoch, and returns `(theta, losses)`.

In [ ]:
# ── Batch Gradient Descent ────────────────────────────────────────────────────
def batch_gd(X, y, lr=0.01, n_epochs=100, seed=42):
    """
    Uses ALL n samples to compute one gradient per epoch.
    Most accurate gradient direction; slowest per epoch on large datasets.
    """
    np.random.seed(seed)
    n, p  = X.shape
    theta = np.zeros(p)        # start from zeros
    losses = []

    for epoch in range(n_epochs):
        residuals = X @ theta - y               # (n,)
        gradient  = (2 / n) * X.T @ residuals   # (p,) exact gradient
        theta    -= lr * gradient
        mse       = np.mean(residuals ** 2)      # record BEFORE update for clarity
        losses.append(mse)

    return theta, np.array(losses)


# ── Stochastic Gradient Descent ───────────────────────────────────────────────
def sgd(X, y, lr=0.01, n_epochs=100, seed=42):
    """
    Uses ONE randomly selected sample per parameter update.
    n updates per epoch. Very noisy but fast.
    Data is shuffled each epoch to avoid cyclic bias.
    """
    np.random.seed(seed)
    n, p  = X.shape
    theta = np.zeros(p)
    losses = []

    for epoch in range(n_epochs):
        idx = np.random.permutation(n)    # shuffle indices each epoch
        for i in idx:
            xi = X[i:i+1]                 # (1, p)
            yi = y[i:i+1]                 # (1,)
            gradient = 2 * xi.T @ (xi @ theta - yi)   # (p,) single-sample gradient
            theta   -= lr * gradient
        # Record full-dataset loss at end of epoch
        mse = np.mean((X @ theta - y) ** 2)
        losses.append(mse)

    return theta, np.array(losses)


# ── Mini-batch Gradient Descent ───────────────────────────────────────────────
def mini_batch_gd(X, y, lr=0.01, n_epochs=100, batch_size=32, seed=42):
    """
    Uses B randomly sampled examples per update.
    n/B updates per epoch. Best balance of accuracy and speed.
    """
    np.random.seed(seed)
    n, p  = X.shape
    theta = np.zeros(p)
    losses = []

    for epoch in range(n_epochs):
        idx = np.random.permutation(n)
        for start in range(0, n, batch_size):
            batch_idx = idx[start : start + batch_size]
            Xb, yb    = X[batch_idx], y[batch_idx]
            gradient  = (2 / len(Xb)) * Xb.T @ (Xb @ theta - yb)
            theta    -= lr * gradient
        mse = np.mean((X @ theta - y) ** 2)
        losses.append(mse)

    return theta, np.array(losses)


print("Functions defined: batch_gd, sgd, mini_batch_gd")
print("Running quick sanity check (5 epochs each)...")
for fn, name in [(batch_gd, 'Batch GD'), (sgd, 'SGD'), (mini_batch_gd, 'Mini-batch GD')]:
    th, ls = fn(X, y, lr=0.01, n_epochs=5)
    print(f"  {name:20s}  final MSE = {ls[-1]:.4f}")

## Section 3 — Convergence Comparison

We run all three optimisers for 150 epochs at the same learning rate and plot loss vs epoch. The signature noise patterns of each variant become immediately visible.

In [ ]:
N_EPOCHS = 150
LR       = 0.05

theta_bgd, losses_bgd  = batch_gd(X, y, lr=LR, n_epochs=N_EPOCHS)
theta_sgd, losses_sgd  = sgd(X, y, lr=LR, n_epochs=N_EPOCHS)
theta_mgd, losses_mgd  = mini_batch_gd(X, y, lr=LR, n_epochs=N_EPOCHS, batch_size=32)

epochs = np.arange(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Linear scale ─────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs, losses_bgd, color='steelblue',  lw=2.5, label='Batch GD')
ax.plot(epochs, losses_mgd, color='seagreen',   lw=2.0, label='Mini-batch GD (B=32)', alpha=0.9)
ax.plot(epochs, losses_sgd, color='tomato',     lw=1.2, label='SGD (B=1)', alpha=0.7)
ax.axhline(loss_opt, color='black', ls='--', lw=1.5, label=f'OLS optimum ({loss_opt:.4f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (full dataset)')
ax.set_title('Convergence Comparison — Linear Scale')
ax.legend(fontsize=9)

# ── Log scale (shows fine structure) ─────────────────────────────────────────
ax = axes[1]
ax.semilogy(epochs, losses_bgd, color='steelblue',  lw=2.5, label='Batch GD')
ax.semilogy(epochs, losses_mgd, color='seagreen',   lw=2.0, label='Mini-batch GD (B=32)', alpha=0.9)
ax.semilogy(epochs, losses_sgd, color='tomato',     lw=1.2, label='SGD (B=1)', alpha=0.7)
ax.axhline(loss_opt, color='black', ls='--', lw=1.5, label=f'OLS optimum')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (log scale)')
ax.set_title('Convergence Comparison — Log Scale')
ax.legend(fontsize=9)

plt.suptitle(f'Gradient Descent Variants — Loss vs Epoch  (lr={LR}, n={N_SAMPLES})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'Variant':<25} {'Final MSE':>12} {'Gap to OLS':>12}")
print("-" * 52)
for name, losses in [('Batch GD', losses_bgd), ('Mini-batch GD', losses_mgd), ('SGD', losses_sgd)]:
    print(f"{name:<25} {losses[-1]:>12.6f} {losses[-1] - loss_opt:>12.6f}")
print(f"{'OLS (analytical)':<25} {loss_opt:>12.6f} {'—':>12}")

## Section 4 — Batch Size Effect

We sweep batch sizes from 1 (pure SGD) to 500 (batch GD) and measure final loss and wall-clock training time. This reveals the sweet spot between noise and efficiency.

In [ ]:
batch_sizes   = [1, 4, 16, 32, 64, 128, 256, 500]
final_losses  = []
training_times = []
N_SWEEP = 80   # enough epochs to show convergence pattern

for B in batch_sizes:
    t0 = time.perf_counter()
    if B == 500:   # full batch
        _, ls = batch_gd(X, y, lr=0.05, n_epochs=N_SWEEP)
    else:
        _, ls = mini_batch_gd(X, y, lr=0.05, n_epochs=N_SWEEP, batch_size=B)
    t1 = time.perf_counter()
    final_losses.append(ls[-1])
    training_times.append((t1 - t0) * 1000)   # ms

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Final loss vs batch size
ax = axes[0]
ax.semilogx(batch_sizes, final_losses, 'o-', color='steelblue', lw=2, ms=8)
ax.axhline(loss_opt, color='black', ls='--', lw=1.5, label='OLS optimum')
ax.set_xlabel('Batch size B (log scale)')
ax.set_ylabel(f'Final MSE after {N_SWEEP} epochs')
ax.set_title('Final Loss vs Batch Size')
ax.set_xticks(batch_sizes)
ax.set_xticklabels([str(b) for b in batch_sizes])
ax.legend()

# Mark extremes
ax.annotate('SGD\n(B=1)\nHigh noise',
            xy=(1, final_losses[0]), xytext=(3, final_losses[0] + 0.1),
            arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato', fontsize=9)
ax.annotate('Batch GD\n(B=500)\nSlowest',
            xy=(500, final_losses[-1]), xytext=(150, final_losses[-1] + 0.1),
            arrowprops=dict(arrowstyle='->', color='grey'), color='grey', fontsize=9)

# Training time vs batch size
ax = axes[1]
ax.semilogx(batch_sizes, training_times, 's--', color='tomato', lw=2, ms=8)
ax.set_xlabel('Batch size B (log scale)')
ax.set_ylabel('Training time (ms)')
ax.set_title('Wall-clock Time vs Batch Size')
ax.set_xticks(batch_sizes)
ax.set_xticklabels([str(b) for b in batch_sizes])

plt.suptitle('Batch Size Trade-off: Noise vs Efficiency', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'Batch Size':>12} {'Final MSE':>12} {'Time (ms)':>12}")
print("-" * 38)
for B, loss, t in zip(batch_sizes, final_losses, training_times):
    print(f"{B:>12} {loss:>12.5f} {t:>12.1f}")

## Section 5 — Learning Rate Sensitivity

Learning rate is the single most important hyperparameter. Too small: painfully slow. Too large: diverges. We sweep `lr ∈ {0.001, 0.01, 0.1}` for all three variants and plot loss trajectories.

In [ ]:
lr_values  = [0.001, 0.01, 0.1]
lr_labels  = ['lr=0.001 (too slow)', 'lr=0.01 (good)', 'lr=0.1 (may diverge)']
lr_colors  = ['steelblue', 'seagreen', 'tomato']
N_LR       = 120

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
variant_fns = [
    ('Batch GD',       lambda lr: batch_gd(X, y, lr=lr, n_epochs=N_LR)),
    ('SGD',            lambda lr: sgd(X, y, lr=lr, n_epochs=N_LR)),
    ('Mini-batch GD',  lambda lr: mini_batch_gd(X, y, lr=lr, n_epochs=N_LR, batch_size=32)),
]

for ax, (variant_name, fn) in zip(axes, variant_fns):
    for lr, label, color in zip(lr_values, lr_labels, lr_colors):
        _, ls = fn(lr)
        # Clip losses to avoid divergent values dominating the y-axis
        ls_clipped = np.clip(ls, 0, loss_opt * 200)
        ax.plot(ls_clipped, color=color, lw=2, label=label)

    ax.axhline(loss_opt, color='black', ls=':', lw=1.5, label='OLS optimum')
    ax.set_title(variant_name, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)

plt.suptitle('Learning Rate Sensitivity — All Three Variants',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observations:")
print("  lr=0.001 : converges but very slowly — may need 10x more epochs")
print("  lr=0.01  : good balance — reaches near-optimum in ~50 epochs")
print("  lr=0.1   : may oscillate or diverge — especially for Batch GD")
print("\nRule of thumb: start with lr=0.01, halve it if loss diverges, double it if convergence is slow.")

## Section 6 — SGD Parameter Path Visualisation

We track how the two most important coefficients (`θ₁` = sqft, `θ₂` = rooms) evolve during SGD training. The scatter plot shows the characteristic **bouncing** around the minimum, contrasted with Batch GD's smooth trajectory.

In [ ]:
def sgd_with_path(X, y, lr=0.01, n_epochs=60, seed=42):
    """SGD that records theta after every single sample update (for path viz)."""
    np.random.seed(seed)
    n, p   = X.shape
    theta  = np.zeros(p)
    path   = [theta.copy()]    # record every update
    losses = []

    for epoch in range(n_epochs):
        idx = np.random.permutation(n)
        for i in idx:
            xi = X[i:i+1]
            yi = y[i:i+1]
            gradient = 2 * xi.T @ (xi @ theta - yi)
            theta   -= lr * gradient
            path.append(theta.copy())
        losses.append(np.mean((X @ theta - y) ** 2))

    return theta, np.array(losses), np.array(path)


def batch_gd_with_path(X, y, lr=0.01, n_epochs=60, seed=42):
    """Batch GD recording theta after every epoch."""
    np.random.seed(seed)
    n, p  = X.shape
    theta = np.zeros(p)
    path  = [theta.copy()]
    losses = []
    for _ in range(n_epochs):
        gradient = (2 / n) * X.T @ (X @ theta - y)
        theta   -= lr * gradient
        path.append(theta.copy())
        losses.append(np.mean((X @ theta - y) ** 2))
    return theta, np.array(losses), np.array(path)


# Collect paths (use indices 1 and 2: sqft and rooms coefficients)
_, losses_sgd_p,  path_sgd  = sgd_with_path(X, y, lr=0.01, n_epochs=40)
_, losses_bgd_p,  path_bgd  = batch_gd_with_path(X, y, lr=0.01, n_epochs=40)

theta1_idx, theta2_idx = 1, 2   # sqft and rooms indices

# Create a loss contour in the θ₁–θ₂ plane (all other params fixed at optimum)
t1_range = np.linspace(theta_opt[theta1_idx] - 3, theta_opt[theta1_idx] + 3, 60)
t2_range = np.linspace(theta_opt[theta2_idx] - 3, theta_opt[theta2_idx] + 3, 60)
T1, T2   = np.meshgrid(t1_range, t2_range)

Z = np.zeros_like(T1)
for i in range(T1.shape[0]):
    for j in range(T1.shape[1]):
        th = theta_opt.copy()
        th[theta1_idx] = T1[i, j]
        th[theta2_idx] = T2[i, j]
        Z[i, j] = np.mean((X @ th - y) ** 2)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, path, name, color in [
    (axes[0], path_sgd,  'SGD',      'tomato'),
    (axes[1], path_bgd,  'Batch GD', 'steelblue'),
]:
    # Contour plot
    cp = ax.contourf(T1, T2, Z, levels=25, cmap='YlOrRd', alpha=0.6)
    ax.contour(T1, T2, Z, levels=25, colors='grey', linewidths=0.4, alpha=0.5)
    plt.colorbar(cp, ax=ax, label='MSE')

    # Parameter path
    # For SGD: thin out to every 50th update to avoid clutter
    step = max(1, len(path) // 300)
    path_thin = path[::step]
    t1_path = path_thin[:, theta1_idx]
    t2_path = path_thin[:, theta2_idx]

    ax.scatter(t1_path, t2_path, c=np.linspace(0, 1, len(t1_path)),
               cmap='Blues', s=8, zorder=3, alpha=0.7)
    ax.plot(t1_path, t2_path, color=color, lw=0.8, alpha=0.5)
    ax.plot(t1_path[0],  t2_path[0],  'go', ms=10, label='Start', zorder=5)
    ax.plot(t1_path[-1], t2_path[-1], 'b*', ms=12, label='End',   zorder=5)
    ax.plot(theta_opt[theta1_idx], theta_opt[theta2_idx], 'k+', ms=15,
            markeredgewidth=3, label='OLS Optimum', zorder=5)

    ax.set_xlabel('θ₁ (sqft coefficient)')
    ax.set_ylabel('θ₂ (rooms coefficient)')
    ax.set_title(f'{name} — Parameter Path in θ-Space', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('SGD Bounces Around the Minimum; Batch GD Glides Directly',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 7 — Learning Rate Scheduling

A fixed learning rate has a fundamental tension: it needs to be large early (fast progress) and small late (precise convergence). **Learning rate schedules** resolve this by decaying the rate over training.

### Three schedules implemented:
| Schedule | Formula | Behaviour |
|---|---|---|
| Fixed | $\eta_t = \eta_0$ | Baseline; simple but suboptimal |
| Step decay | $\eta_t = \eta_0 \times 0.5^{\lfloor t/10 \rfloor}$ | Halves every 10 epochs |
| Cosine annealing | $\eta_t = \eta_{min} + \frac{\eta_0 - \eta_{min}}{2}(1 + \cos(\pi t / T))$ | Smooth decay following cosine curve |

In [ ]:
# ── Learning rate schedule functions ─────────────────────────────────────────
def lr_fixed(epoch, lr0=0.05, **kw):
    return lr0

def lr_step_decay(epoch, lr0=0.05, drop=0.5, epochs_drop=10, **kw):
    """Halves every `epochs_drop` epochs."""
    return lr0 * (drop ** (epoch // epochs_drop))

def lr_cosine_annealing(epoch, lr0=0.05, lr_min=1e-4, T_max=100, **kw):
    """Smooth cosine decay from lr0 to lr_min over T_max epochs."""
    return lr_min + 0.5 * (lr0 - lr_min) * (1 + np.cos(np.pi * epoch / T_max))


# ── SGD with schedule ─────────────────────────────────────────────────────────
def sgd_with_schedule(X, y, lr_fn, n_epochs=100, batch_size=32, seed=42, **lr_kwargs):
    """Mini-batch GD that applies a learning-rate schedule each epoch."""
    np.random.seed(seed)
    n, p  = X.shape
    theta = np.zeros(p)
    losses = []
    lr_history = []

    for epoch in range(n_epochs):
        lr = lr_fn(epoch, **lr_kwargs)           # current learning rate
        lr_history.append(lr)
        idx = np.random.permutation(n)
        for start in range(0, n, batch_size):
            batch_idx = idx[start : start + batch_size]
            Xb, yb    = X[batch_idx], y[batch_idx]
            gradient  = (2 / len(Xb)) * Xb.T @ (Xb @ theta - yb)
            theta    -= lr * gradient
        losses.append(np.mean((X @ theta - y) ** 2))

    return theta, np.array(losses), np.array(lr_history)


N_SCHED = 100
lr0     = 0.10   # larger initial lr to show the effect clearly

_, ls_fixed,  lrs_fixed  = sgd_with_schedule(X, y, lr_fixed,          n_epochs=N_SCHED, lr0=lr0)
_, ls_step,   lrs_step   = sgd_with_schedule(X, y, lr_step_decay,     n_epochs=N_SCHED, lr0=lr0)
_, ls_cosine, lrs_cosine = sgd_with_schedule(X, y, lr_cosine_annealing, n_epochs=N_SCHED,
                                              lr0=lr0, T_max=N_SCHED)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# ── Top panel: loss curves ───────────────────────────────────────────────────
for ls, label, color in [
    (ls_fixed,  'Fixed lr=0.10',    'steelblue'),
    (ls_step,   'Step decay',       'seagreen'),
    (ls_cosine, 'Cosine annealing', 'darkorange'),
]:
    ax1.plot(ls, lw=2, label=label, color=color)
ax1.axhline(loss_opt, color='black', ls='--', lw=1.5, label='OLS optimum')
ax1.set_ylabel('MSE')
ax1.set_title('Loss Curves with Different LR Schedules', fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim(bottom=0)

# ── Bottom panel: learning rate schedules ────────────────────────────────────
for lrs, label, color in [
    (lrs_fixed,  'Fixed',           'steelblue'),
    (lrs_step,   'Step decay',      'seagreen'),
    (lrs_cosine, 'Cosine annealing','darkorange'),
]:
    ax2.plot(lrs, lw=2, label=label, color=color)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning rate (lr)')
ax2.set_title('Learning Rate Schedule', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Final MSE  | Fixed: {ls_fixed[-1]:.5f} | Step decay: {ls_step[-1]:.5f} | Cosine: {ls_cosine[-1]:.5f}")
print(f"Final lr   | Fixed: {lrs_fixed[-1]:.5f} | Step decay: {lrs_step[-1]:.5f} | Cosine: {lrs_cosine[-1]:.5f}")

## Section 8 — Comparing to sklearn's SGDRegressor

We verify our scratch implementation against sklearn's `SGDRegressor` by matching hyperparameters and checking that loss trajectories are comparable. Small differences arise from sklearn's internal implementation details (e.g., regularisation, averaging), but the overall behaviour should match.

In [ ]:
from sklearn.linear_model import SGDRegressor

# ── Our scratch SGD ───────────────────────────────────────────────────────────
LR_MATCH   = 0.01
N_MATCH    = 100
_, losses_scratch, _ = sgd_with_path(X, y, lr=LR_MATCH, n_epochs=N_MATCH)

# ── sklearn SGDRegressor — match parameters as closely as possible ───────────
# eta0 = our lr; penalty='none' = no regularisation;
# learning_rate='constant' = fixed lr (no schedule); shuffle=True per epoch
sklearn_losses = []
theta_sk = np.zeros(X.shape[1])

# We use partial_fit to record epoch-by-epoch loss
sgd_sk = SGDRegressor(
    loss='squared_error',
    learning_rate='constant',
    eta0=LR_MATCH,
    penalty='l2',          # small L2 for numerical stability
    alpha=1e-9,
    shuffle=True,
    random_state=42,
    max_iter=1,            # one epoch at a time
    tol=None,
    warm_start=True,
)

for epoch in range(N_MATCH):
    sgd_sk.fit(X, y)    # warm_start=True means it continues from last state
    y_pred = sgd_sk.predict(X)
    sklearn_losses.append(mean_squared_error(y, y_pred))

# ── Plot ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(losses_scratch,  color='steelblue', lw=2.5, label='Scratch SGD (our implementation)')
plt.plot(sklearn_losses,  color='tomato',    lw=2.0, ls='--', label='sklearn SGDRegressor')
plt.axhline(loss_opt, color='black', ls=':', lw=1.5, label=f'OLS optimum ({loss_opt:.4f})')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title(f'Scratch SGD vs sklearn SGDRegressor  (lr={LR_MATCH}, no regularisation)',
          fontweight='bold')
plt.legend()
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f"Final MSE — scratch SGD   : {losses_scratch[-1]:.6f}")
print(f"Final MSE — sklearn SGD   : {sklearn_losses[-1]:.6f}")
print(f"OLS optimal MSE           : {loss_opt:.6f}")
print("\nNote: small differences are expected due to sklearn's mini-optimisations.")
print("The key check: both converge to similar loss values near the OLS optimum.")

## Section 9 — Summary: Choosing the Right Variant

A consolidated view of all convergence patterns studied.

In [ ]:
# ── Reconstruct clean comparison at identical conditions ─────────────────────
N_FINAL = 150
LR_FINAL = 0.05

_, losses_bgd_f  = batch_gd(X, y, lr=LR_FINAL, n_epochs=N_FINAL)
_, losses_sgd_f  = sgd(X, y,       lr=LR_FINAL, n_epochs=N_FINAL)
_, losses_m16_f  = mini_batch_gd(X, y, lr=LR_FINAL, n_epochs=N_FINAL, batch_size=16)
_, losses_m32_f  = mini_batch_gd(X, y, lr=LR_FINAL, n_epochs=N_FINAL, batch_size=32)
_, losses_m64_f  = mini_batch_gd(X, y, lr=LR_FINAL, n_epochs=N_FINAL, batch_size=64)

fig, ax = plt.subplots(figsize=(11, 5))

plot_configs = [
    (losses_bgd_f, 'Batch GD (B=500)',      'royalblue',  2.5, '-'),
    (losses_m64_f, 'Mini-batch GD (B=64)',  'seagreen',   2.0, '-'),
    (losses_m32_f, 'Mini-batch GD (B=32)',  'mediumorchid', 2.0, '-.'),
    (losses_m16_f, 'Mini-batch GD (B=16)',  'darkorange', 1.8, '--'),
    (losses_sgd_f, 'SGD (B=1)',             'tomato',     1.3, '-'),
]

for ls, label, color, lw, ls_style in plot_configs:
    ax.plot(ls, color=color, lw=lw, ls=ls_style, label=label, alpha=0.85)

ax.axhline(loss_opt, color='black', ls=':', lw=2, label=f'OLS optimum ({loss_opt:.4f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title(f'All Variants — Convergence Summary  (lr={LR_FINAL})', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

# ── Decision table ────────────────────────────────────────────────────────────
table = pd.DataFrame({
    'Variant':         ['Batch GD', 'SGD (B=1)', 'Mini-batch (B=32)'],
    'Gradient quality':['Exact',    'Very noisy','Good approx'],
    'Updates/epoch':   ['1',        'n=500',     'n/B ≈ 16'],
    'Convergence':     ['Smooth',   'Noisy',     'Smooth-ish'],
    'Best use case':   ['Small data', 'Online/streaming', 'Deep learning standard'],
    'Final MSE':       [f'{losses_bgd_f[-1]:.4f}', f'{losses_sgd_f[-1]:.4f}', f'{losses_m32_f[-1]:.4f}']
})
print(table.to_string(index=False))

## Section 10 — Key Takeaways

### Gradient descent variants in one sentence each

- **Batch GD:** Compute the exact gradient on all data → one step per epoch → smooth but slow on large datasets.
- **SGD:** Compute a noisy gradient on one sample → n steps per epoch → fast but bounces around the minimum.
- **Mini-batch GD:** The sweet spot → B samples → n/B steps per epoch → standard in deep learning.

### Learning rate rules of thumb

| Symptom | Diagnosis | Fix |
|---|---|---|
| Loss oscillates / diverges | Learning rate too high | Halve lr |
| Loss barely decreases | Learning rate too low | Double lr |
| Fast early, stalls late | No schedule | Add step decay or cosine annealing |
| SGD never fully converges | Constant lr | Decay lr as training progresses |

### The practitioner's checklist

1. Always **standardise features** before gradient descent
2. **Shuffle data** every epoch (especially for SGD)
3. Plot **loss vs epoch** before anything else — diagnose divergence early
4. Start with **batch size 32** and lr around 0.01; tune from there
5. Use a **learning rate schedule** for final training runs
6. Compare against the **analytical solution** (OLS) to verify your implementation

## Self-Check Questions

Answer each question on your own, then expand the answer.

---

**Q1.** SGD with lr = 0.01 doesn't converge to exactly the minimum — it bounces around it. What two things can you do to help it converge better?

<details>
<summary>Show answer</summary>

**Two strategies:**

1. **Decay the learning rate over time.** A fixed learning rate causes permanent oscillation because each noisy gradient step is equally sized. By *decaying* the lr (step decay, cosine annealing, or `1/t` decay), the steps become smaller as you get closer to the minimum, allowing you to settle precisely. This is the most effective fix.

2. **Increase the batch size (switch to mini-batch GD).** Using B > 1 samples per update averages out the noise in the gradient estimate. With B = 32, the gradient direction is more accurate than B = 1, so the path is less erratic. You lose the "one sample per update" speed advantage but gain a smoother, more convergent trajectory.

**Bonus:** averaging the parameter values over later epochs (*Polyak-Ruppert averaging*) also helps — the mean of the noisy path is closer to the true minimum than any single point on it.

</details>

---

**Q2.** Batch size B = 1 (SGD) does n = 500 parameter updates per epoch, while B = 32 (mini-batch) does only 500/32 ≈ 16 updates per epoch. Yet mini-batch usually converges in fewer epochs. Why?

<details>
<summary>Show answer</summary>

**Answer: gradient quality matters more than update count.**

Each SGD update uses a single noisy sample. Many of those 500 updates are *misleading* — they point in the wrong direction due to noise, effectively wasting computation. The model zigzags rather than progressing toward the minimum.

Each mini-batch update (B = 32) uses an average gradient over 32 samples — a much better estimate of the true gradient direction. These 16 updates per epoch are *more productive*, so the model makes more genuine progress per epoch.

Analogy: taking 500 small erratic steps (SGD) vs 16 confident strides in the right direction (mini-batch). The confident strides cover more real ground even though there are fewer of them.

**In practice:** mini-batch GD often converges in fewer epochs, and since each update is still cheap (only B samples), the total computation is much less than batch GD.

</details>

---

**Q3.** Your batch gradient descent training loss is oscillating and not decreasing. What is the most likely cause? What's the fix?

<details>
<summary>Show answer</summary>

**Most likely cause: learning rate is too high.**

Here's the geometry: the loss surface is a bowl (for convex problems like linear regression). With a small lr, each step moves partway down the bowl. With a large lr, each step *overshoots* the bottom and lands on the far wall — higher than where you started. The next step overshoots back. The result: the loss oscillates and may even *increase* over time.

**Visualisation:** imagine rolling a ball down a bowl. With light tap → rolls to bottom. With strong kick → flies back and forth across the bowl forever.

**The fix:**
1. **Reduce the learning rate** — try halving it (e.g. 0.1 → 0.05 → 0.01) until loss decreases monotonically.
2. **Plot loss vs epoch after every change** — a monotonically decreasing curve confirms a good lr.
3. If you want a more systematic approach, use a **learning rate range test**: train for a few epochs with lr increasing from 1e-6 to 1.0 and plot loss vs lr. The optimal lr is just before the loss starts increasing.

**Secondary causes** (less common): un-standardised features with very different scales can cause oscillation even at moderate lr. Always standardise features before gradient descent.

</details>

In [ ]:
# ── Bonus: Learning Rate Range Test (LR Finder) ───────────────────────────────
# Sweep lr from very small to very large; plot loss. Optimal lr is just before
# the loss starts climbing — a widely-used technique in deep learning.

np.random.seed(42)

lr_test_values = np.logspace(-4, 1, 80)   # from 0.0001 to 10
test_losses    = []

for lr_test in lr_test_values:
    # Run just 3 epochs at each lr value to get a quick loss estimate
    _, ls_test = mini_batch_gd(X, y, lr=lr_test, n_epochs=3, batch_size=32, seed=42)
    # Clip to avoid overflow from divergent learning rates
    test_losses.append(min(ls_test[-1], loss_opt * 500))

# Smooth for readability
def smooth(arr, window=5):
    return np.convolve(arr, np.ones(window)/window, mode='same')

test_losses_smooth = smooth(np.array(test_losses))

# Find the lr at the steepest negative slope (biggest loss drop)
gradients  = np.gradient(test_losses_smooth)
best_lr_idx = np.argmin(gradients)
best_lr     = lr_test_values[best_lr_idx]

plt.figure(figsize=(9, 4))
plt.semilogx(lr_test_values, test_losses,        color='lightgrey', lw=1.5, label='Raw loss')
plt.semilogx(lr_test_values, test_losses_smooth, color='steelblue', lw=2.5, label='Smoothed loss')
plt.axvline(best_lr, color='tomato', ls='--', lw=2,
            label=f'Suggested lr ≈ {best_lr:.4f} (steepest descent)')
plt.xlabel('Learning rate (log scale)')
plt.ylabel('MSE after 3 epochs')
plt.title('Learning Rate Range Test — Find the Sweet Spot', fontweight='bold')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Suggested starting lr: {best_lr:.4f}")
print("Use a lr just before the loss starts rising steeply — that's your upper bound.")
print("For mini-batch GD, the optimal lr is typically in the range [best_lr/10, best_lr].")